In [0]:
from pyspark.sql.functions import regexp_replace, col

In [0]:
# ============================================
# 1. Employees (from Postgres, via ADF)
# ============================================
df_employees = spark.read.option("header", "true").csv(
    "abfss://raw@hr1storage1.dfs.core.windows.net/employees/"
)

# ============================================
# 2. Attendees (CSV, manual upload)
# ============================================
df_attendees = spark.read.option("header", "true").csv(
    "abfss://raw@hr1storage1.dfs.core.windows.net/attendees/"
)

# ============================================
# 3. Performance (Excel, manual upload)
#    Requires the spark-excel library installed on the cluster first:
#    Compute -> your cluster -> Libraries -> Install new -> Maven -> "com.crealytics:spark-excel"
# ============================================
df_performance = spark.read.format("com.crealytics.spark.excel") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("abfss://raw@hr1storage1.dfs.core.windows.net/performance/performance_reviews.xlsx")

# ============================================
# Quick check all three loaded correctly
# ============================================
print("Employees:")
df_employees.printSchema()
print(f"Row count: {df_employees.count()}\n")

print("Attendees:")
df_attendees.printSchema()
print(f"Row count: {df_attendees.count()}\n")

print("Performance:")
df_performance.printSchema()
print(f"Row count: {df_performance.count()}\n")

Employees:
root
 |-- EmployeeID: string (nullable = true)
 |-- Name: string (nullable = true)
 |-- Department: string (nullable = true)
 |-- Position: string (nullable = true)
 |-- HireDate: string (nullable = true)
 |-- Salary: string (nullable = true)

Row count: 21

Attendees:
root
 |-- EmployeeID: string (nullable = true)
 |-- Month: string (nullable = true)
 |-- DaysPresent: string (nullable = true)
 |-- DaysAbsent: string (nullable = true)
 |-- LateArrivals: string (nullable = true)

Row count: 120

Performance:
root
 |-- EmployeeID: double (nullable = true)
 |-- ReviewDate: timestamp (nullable = true)
 |-- PerformanceScore: string (nullable = true)
 |-- ManagerComment: string (nullable = true)

Row count: 20



In [0]:

df_attendees_clean = df_attendees.withColumn(
    "EmployeeID",
    regexp_replace(col("EmployeeID"), "EMP", "")
)

# Verify the fix worked
df_attendees_clean.groupBy("EmployeeID").count().orderBy("EmployeeID").show(25)

+----------+-----+
|EmployeeID|count|
+----------+-----+
|      1001|    6|
|      1002|    6|
|      1003|    6|
|      1004|    6|
|      1005|    6|
|      1006|    6|
|      1007|    6|
|      1008|    6|
|      1009|    6|
|      1010|    6|
|      1011|    6|
|      1012|    6|
|      1013|    6|
|      1014|    6|
|      1015|    6|
|      1016|    6|
|      1017|    6|
|      1018|    6|
|      1019|    6|
|      1020|    6|
+----------+-----+



In [0]:
df_attendees_clean.printSchema()

root
 |-- EmployeeID: string (nullable = true)
 |-- Month: string (nullable = true)
 |-- DaysPresent: string (nullable = true)
 |-- DaysAbsent: string (nullable = true)
 |-- LateArrivals: string (nullable = true)



In [0]:
from pyspark.sql.functions import coalesce, lit, col, when

df_attendees_clean = df_attendees_clean.withColumn(
    "DaysAbsent_flagged_invalid",
    coalesce(col("DaysAbsent").cast("int") < 0, lit(False))
).withColumn(
    "DaysAbsent",
    when(col("DaysAbsent").cast("int") < 0, None).otherwise(col("DaysAbsent").cast("int"))
)

In [0]:
from pyspark.sql.functions import expr, coalesce, trim, col

df_employees_clean = df_employees
for c in df_employees.columns:
    df_employees_clean = df_employees_clean.withColumn(c, trim(col(c)))

df_employees_clean = df_employees_clean.withColumn(
    "HireDate_parsed",
    coalesce(
        expr("try_to_date(HireDate, 'yyyy-MM-dd')"),
        expr("try_to_date(HireDate, 'MM/dd/yyyy')"),
        expr("try_to_date(HireDate, 'dd-MM-yyyy')")
    )
)

# Verify nothing failed to parse
df_employees_clean.filter(col("HireDate_parsed").isNull() & col("HireDate").isNotNull()) \
    .select("HireDate", "HireDate_parsed").show(20, truncate=False)

+--------+---------------+
|HireDate|HireDate_parsed|
+--------+---------------+
+--------+---------------+



In [0]:
df_performance.select("ReviewDate").distinct().orderBy("ReviewDate").show(25, truncate=False)

+-------------------+
|ReviewDate         |
+-------------------+
|NULL               |
|2026-03-31 00:00:00|
+-------------------+



In [0]:
df_attendees_clean.select("Month").distinct().orderBy("Month").show(25, truncate=False)

+--------+
|Month   |
+--------+
|01      |
|02      |
|03      |
|04      |
|05      |
|06      |
|Apr     |
|April   |
|Feb     |
|February|
|January |
|Jun     |
|June    |
|Mar     |
|March   |
|May     |
+--------+



In [0]:
df_performance.groupBy("ReviewDate").count().show()

+-------------------+-----+
|         ReviewDate|count|
+-------------------+-----+
|2026-03-31 00:00:00|   18|
|               NULL|    2|
+-------------------+-----+



In [0]:
from pyspark.sql.functions import when, col

df_attendees_clean = df_attendees_clean.withColumn(
    "Month_standardized",
    when(col("Month").isin("01","1","Jan","January"), 1)
    .when(col("Month").isin("02","2","Feb","February"), 2)
    .when(col("Month").isin("03","3","Mar","March"), 3)
    .when(col("Month").isin("04","4","Apr","April"), 4)
    .when(col("Month").isin("05","5","May"), 5)
    .when(col("Month").isin("06","6","Jun","June"), 6)
    .when(col("Month").isin("07","7","Jul","July"), 7)
    .when(col("Month").isin("08","8","Aug","August"), 8)
    .when(col("Month").isin("09","9","Sep","September"), 9)
    .when(col("Month").isin("10","Oct","October"), 10)
    .when(col("Month").isin("11","Nov","November"), 11)
    .when(col("Month").isin("12","Dec","December"), 12)
    .otherwise(None)
)

# Verify nothing fell through uncaught
df_attendees_clean.filter(col("Month_standardized").isNull()).select("Month").distinct().show()

+-----+
|Month|
+-----+
+-----+



In [0]:
from pyspark.sql.functions import trim, col, when, regexp_extract, avg, expr


# ============================================
# Performance: extract numeric scores safely with try_cast
# ============================================
df_performance = df_performance.withColumn(
    "PerformanceScore_numeric",
    expr("try_cast(regexp_extract(PerformanceScore, '^[0-9]+\\\\.?[0-9]*$', 0) as double)")
)

# Compute the average of only the VALID numeric scores (nulls auto-excluded)
avg_score = df_performance.select(avg("PerformanceScore_numeric")).collect()[0][0]
print(f"Average PerformanceScore: {avg_score}")

# Flag which rows were imputed, then fill
df_performance = df_performance.withColumn(
    "PerformanceScore_imputed", col("PerformanceScore_numeric").isNull()
).fillna({"PerformanceScore_numeric": avg_score})

# Quick check
df_performance.select("PerformanceScore", "PerformanceScore_numeric", "PerformanceScore_imputed").show(30, truncate=False)

Average PerformanceScore: 3.56
+----------------+------------------------+------------------------+
|PerformanceScore|PerformanceScore_numeric|PerformanceScore_imputed|
+----------------+------------------------+------------------------+
|3.6             |3.6                     |false                   |
|4.3             |4.3                     |false                   |
|high            |3.56                    |true                    |
|4.5             |4.5                     |false                   |
|3.7             |3.7                     |false                   |
|3.6             |3.6                     |false                   |
|                |3.56                    |true                    |
|high            |3.56                    |true                    |
|4.6             |4.6                     |false                   |
|3.2             |3.2                     |false                   |
|3.1             |3.1                     |false                   |
|N/

In [0]:
df_employees_clean = df_employees_clean.fillna({"Department": "MISSING"})

# Verify
df_employees_clean.filter(col("Department") == "MISSING").select("EmployeeID", "Name", "Department").show()

+----------+-------------+----------+
|EmployeeID|         Name|Department|
+----------+-------------+----------+
|      1007| Michael Reid|   MISSING|
|      1012|Benjamin Cruz|   MISSING|
+----------+-------------+----------+



In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, trim

print("=" * 60)
print("EMPLOYEES")
print("=" * 60)

print("Row count:", df_employees_clean.count())

print("\n-- Nulls/empties per column --")
df_employees_clean.select([
    F.count(F.when(F.col(c).isNull() | (F.trim(F.col(c).cast("string")) == ""), c)).alias(c)
    for c in df_employees_clean.columns
]).show()

print("-- HireDate parsing: any failures? --")
df_employees_clean.filter(col("HireDate_parsed").isNull() & col("HireDate").isNotNull()).count()

print("-- Whitespace check --")
for c in df_employees_clean.columns:
    dirty = df_employees_clean.filter(col(c).cast("string") != trim(col(c).cast("string"))).count()
    if dirty > 0:
        print(f"⚠️  '{c}' still has {dirty} rows with whitespace")
print("(no warnings above = clean)")

print("\n" + "=" * 60)
print("ATTENDEES")
print("=" * 60)

print("Row count:", df_attendees_clean.count())

print("\n-- Nulls/empties per column --")
df_attendees_clean.select([
    F.count(F.when(F.col(c).isNull() | (F.trim(F.col(c).cast("string")) == ""), c)).alias(c)
    for c in df_attendees_clean.columns
]).show()

print("-- EmployeeID format check (should all be plain numbers, no EMP prefix) --")
df_attendees_clean.filter(col("EmployeeID").rlike("[A-Za-z]")).select("EmployeeID").distinct().show()
print("(empty above = no more EMP prefixes)")

print("-- Month_standardized: any values 1-12 missing/out of range? --")
df_attendees_clean.filter(
    col("Month_standardized").isNull() | (col("Month_standardized") < 1) | (col("Month_standardized") > 12)
).select("Month", "Month_standardized").distinct().show()
print("(empty above = all months mapped correctly)")

print("-- DaysAbsent: any negatives remaining? --")
df_attendees_clean.filter(col("DaysAbsent") < 0).count()

print("-- DaysAbsent_flagged_invalid: how many rows flagged? --")
df_attendees_clean.filter(col("DaysAbsent_flagged_invalid") == True).count()

print("\n" + "=" * 60)
print("PERFORMANCE")
print("=" * 60)

print("Row count:", df_performance.count())

print("\n-- Nulls/empties per column --")
df_performance.select([
    F.count(F.when(F.col(c).isNull() | (F.trim(F.col(c).cast("string")) == ""), c)).alias(c)
    for c in df_performance.columns
]).show()

print("-- PerformanceScore_numeric: any nulls left (should be 0, all filled with avg)? --")
df_performance.filter(col("PerformanceScore_numeric").isNull()).count()

print("-- How many rows were imputed with the average? --")
df_performance.filter(col("PerformanceScore_imputed") == True).count()

print("-- ReviewDate: still 2 legitimate nulls, confirm nothing else broke --")
df_performance.groupBy(col("ReviewDate").isNull().alias("is_null")).count().show()



EMPLOYEES
Row count: 21

-- Nulls/empties per column --
+----------+----+----------+--------+--------+------+---------------+
|EmployeeID|Name|Department|Position|HireDate|Salary|HireDate_parsed|
+----------+----+----------+--------+--------+------+---------------+
|         0|   0|         0|       0|       0|     0|              0|
+----------+----+----------+--------+--------+------+---------------+

-- HireDate parsing: any failures? --
-- Whitespace check --
(no warnings above = clean)

ATTENDEES
Row count: 120

-- Nulls/empties per column --
+----------+-----+-----------+----------+------------+--------------------------+------------------+
|EmployeeID|Month|DaysPresent|DaysAbsent|LateArrivals|DaysAbsent_flagged_invalid|Month_standardized|
+----------+-----+-----------+----------+------------+--------------------------+------------------+
|         0|    0|          0|        15|          15|                         0|                 0|
+----------+-----+-----------+----------+-

In [0]:
print("HireDate parsing failures:", df_employees_clean.filter(col("HireDate_parsed").isNull() & col("HireDate").isNotNull()).count())

print("DaysAbsent negatives remaining:", df_attendees_clean.filter(col("DaysAbsent") < 0).count())
print("DaysAbsent flagged invalid:", df_attendees_clean.filter(col("DaysAbsent_flagged_invalid") == True).count())

print("PerformanceScore_numeric nulls remaining:", df_performance.filter(col("PerformanceScore_numeric").isNull()).count())
print("PerformanceScore imputed count:", df_performance.filter(col("PerformanceScore_imputed") == True).count())

HireDate parsing failures: 0
DaysAbsent negatives remaining: 0
DaysAbsent flagged invalid: 15
PerformanceScore_numeric nulls remaining: 0
PerformanceScore imputed count: 5


In [0]:
df_attendees_clean.groupBy("DaysAbsent_flagged_invalid").count().show()

+--------------------------+-----+
|DaysAbsent_flagged_invalid|count|
+--------------------------+-----+
|                      true|   15|
|                     false|  105|
+--------------------------+-----+



In [0]:
# Remove duplicate EmployeeIDs from employee dimension
df_employees_clean = df_employees_clean.dropDuplicates(["EmployeeID"])

# Write employees to Silver
df_employees_clean.write.mode("overwrite").format("delta") \
    .save("abfss://silver@hr1storage1.dfs.core.windows.net/employees/")


# Write attendance to Silver
df_attendees_clean.write.mode("overwrite").format("delta") \
    .save("abfss://silver@hr1storage1.dfs.core.windows.net/attendees/")


# Write performance to Silver
df_performance.write.mode("overwrite").format("delta") \
    .save("abfss://silver@hr1storage1.dfs.core.windows.net/performance/")